# Tipado y modelos de datos con Pydantic

**Unidad 1 · Sesión 2**

Anotaciones de tipo, análisis estático y validación de datos en Python.

## Contenido

| Sección | Tema |
|---|---|
| 0 | Preparación |
| 1 | Anotaciones de tipo y colecciones |
| 2 | Análisis estático con mypy |
| Ejemplo | Descuentos y precisión del tipo de retorno |
| 3 | Modelos con Pydantic |
| 4 | Restricciones y errores de validación |
| 5 | Validadores de campos |
| 6 | Configuración |
| Ejemplo | Registro de usuarios |
| 7 | Modelos anidados y serialización |
| 8 | Módulo reutilizable |
| 9 | Procesamiento de registros |
| 10 | Una API pequeña con FastAPI y Pydantic |

## Objetivos

- Anotar funciones y colecciones e interpretar diagnósticos de mypy.
- Definir modelos con restricciones y validadores.
- Localizar errores en datos de entrada.
- Serializar modelos anidados y utilizarlos desde un módulo.
- Observar la validación de una petición a una API.

## 0. Preparación

>**mypy** es un verificador estático: analiza el código y contrasta las operaciones
con sus tipos declarados. **Pydantic** es una biblioteca que utiliza anotaciones
para validar y convertir datos durante la ejecución.

Ejecuta la siguiente celda en Colab o Jupyter. Desde terminal puedes instalar las
mismas dependencias con `python -m pip install -r requirements.txt`.

In [3]:
%pip install -q pydantic==2.12.5 mypy==1.19.1 email-validator==2.3.0 fastapi==0.141.1 httpx2==2.13.1

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import sys
from importlib.metadata import version

assert sys.version_info >= (3, 12), "Use Python 3.12 or newer"
print(f"Python: {sys.version.split()[0]}")
print(f"Pydantic: {version('pydantic')}")
print(f"mypy: {version('mypy')}")

Python: 3.14.5
Pydantic: 2.12.5
mypy: 1.19.1


## 1. Del comportamiento al contrato

Una anotación indica el tipo esperado. En `score: float`, `score` es el nombre y
`float` el tipo. Después de los parámetros, `-> str` describe el valor de retorno.
La implementación sigue siendo necesaria: los tipos no calculan el resultado.

**Las anotaciones no validan ni convierten automáticamente los argumentos cuando
Python ejecuta una función.**

In [5]:
def repeat_text(text: str, times: int) -> str:
    return text * times


print(repeat_text("hello ", 2))
# Intentional mismatch: Python executes this even though the annotation says str.
print(repeat_text(3, 2))
assert repeat_text(3, 2) == 6

hello hello 
6


### Análisis estático y ejecución

Las anotaciones describen los tipos esperados de variables, parámetros y retornos.
Mypy utiliza esa información para detectar operaciones incompatibles sin ejecutar
el programa. Las comprobaciones con `assert` verifican resultados concretos cuando
Python sí ejecuta el código.

Estas herramientas tienen alcances diferentes. Una función puede superar el
análisis estático y recibir después un dato externo inválido. El siguiente ejemplo
analiza con mypy la misma llamada que Python aceptó en la celda anterior.

Aquí usamos `mypy_api.run` para analizar texto sin crear un archivo: `-c` pasa
ese texto. `--strict` añade reglas, como exigir funciones anotadas;
`--no-incremental` evita reutilizar resultados de la caché. Más adelante
veremos el comando habitual
sobre un archivo `.py`.

In [6]:
from mypy import api as mypy_api

repeat_source = """def repeat_text(text: str, times: int) -> str:
    return text * times

result = repeat_text(3, 2)
"""
stdout, stderr, exit_status = mypy_api.run(
    ["--strict", "--no-incremental", "-c", repeat_source]
)
print(stdout)
assert exit_status == 1
assert "arg-type" in stdout

runtime_result = repeat_text(3, 2)
print(f"Runtime result: {runtime_result!r}")
assert runtime_result == 6

<string>:4: error: Argument 1 to "repeat_text" has incompatible type "int"; expected "str"  [arg-type]
Found 1 error in 1 file (checked 1 source file)

Runtime result: 6


### Colecciones y valores que pueden faltar

`list[float]` describe una lista de números; `dict[str, int]`, un diccionario de
claves de texto y valores enteros. `float | None` permite un número o ausencia.
No significa «cero». Un argumento con `= None` además puede omitirse en la llamada.

In [7]:
scores: list[float] = [0.0, 0.5, 1.0]
counts: dict[str, int] = {"positive": 2, "neutral": 1}
coordinates: tuple[float, float] = (20.97, -89.62)


def mean_score(values: list[float]) -> float | None:
    if not values:
        return None
    return sum(values) / len(values)


assert mean_score([]) is None
assert mean_score([0.0]) == 0.0
print(mean_score(scores))


def resolve_threshold(threshold: float | None = None) -> float:
    if threshold is None:
        return 0.5
    return threshold


assert resolve_threshold() == 0.5  # The argument is omitted.
assert resolve_threshold(None) == 0.5  # None is passed explicitly.
assert resolve_threshold(0.0) == 0.0  # Zero is a value, not absence.

0.5


- [x] terminado
### <span style="color: red;">Ejercicio 1 · Anotar sin cambiar el comportamiento</span>

Anota `select_scores`: recibe una lista de floats y un umbral float; devuelve
una lista de floats. Conserva los valores iguales al umbral. Comprueba lista vacía,
umbral cero y `[0.5, 0.9, 1.0]` con umbral 0.9.

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [8]:
# Add annotations and checks.
def select_scores(values : list[float], threshold: float)->list[float]:
    return [value for value in values if value >= threshold]

assert select_scores([],0.0)==[]
assert select_scores([0.0], 0.0) == [0.0]
assert isinstance(
    select_scores([0.5,0.9,1.0],0.9),list
), "lista retornada"
assert select_scores([0.5,0.9,1.0],0.9) == [0.9, 1.0]

## 2. Leer un error del verificador

Esta celda analiza otra cadena de código sin ejecutarla ni crear archivos.
`mypy_api.run` devuelve salida, errores internos y código de salida. El código
1 es esperado aquí: hemos
introducido dos errores deliberados.

Lee el mensaje en este orden: ubicación, operación, tipo recibido y tipo esperado.
No cambies un tipo para callar al verificador: revisa qué comportamiento necesitas.

In [9]:
fixed = """def apply_threshold(score: float, threshold: float) -> bool:
    return score >= threshold

result = apply_threshold(0.9, 0.8)

def display_score(score: float ) -> str:
    return f"{score * 100:.1f}%"
"""
stdout, stderr, exit_status = mypy_api.run(
    ["--strict", "--no-incremental", "-c", fixed]
)
print(stdout)
assert exit_status == 0
#assert "arg-type" in stdout
#assert "operator" in stdout

Success: no issues found in 1 source file



- [x] terminado
### <span style="color: red;">Ejercicio 2 · Corregir el contrato</span>

Corrige los dos problemas de `broken_source`:
- convierte explícitamente el texto a float
- y atiende `None` antes de multiplicar.
- Trabaja en otra cadena para conservar el ejemplo defectuoso.

- Ejecuta mypy y comprueba código de salida 0. ¿Por qué convertir
con `float` todavía puede fallar en tiempo de ejecución?

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [10]:
# Write fixed_source and run mypy_api.run on it.
fixed = """def apply_threshold(score: float, threshold: float) -> bool:
    return score >= threshold

result = apply_threshold(0.9, 0.8)

def display_score(score: float ) -> str:
    return f"{score * 100:.1f}%"
"""
stdout, stderr, exit_status = mypy_api.run(
    ["--strict", "--no-incremental", "-c", fixed]
)
print(stdout)
assert exit_status == 0

Success: no issues found in 1 source file



### Precisión del tipo de retorno

El tipo de retorno determina qué operaciones puede comprobar el verificador.
`Iterable[float]` garantiza que los valores pueden recorrerse, pero no garantiza
`len()` ni acceso por índice.

La función siguiente devuelve una lista y declara `list[float]` como retorno.
`Iterable` describe la entrada sin exigir que sea una lista. Su funcionamiento
se estudiará con detalle en la próxima sesión.

In [11]:
from collections.abc import Iterable


def calculate_discounts(items: Iterable[float], discount: float) -> list[float]:
    return [item * (1 - discount) for item in items]


prices = [100.0, 200.0, 300.0]
discounted_prices = calculate_discounts(prices, 0.2)
print(len(discounted_prices))
print(discounted_prices[0])
assert discounted_prices == [80.0, 160.0, 240.0]

3
80.0


Ahora declaramos `Iterable[float]` como retorno, aunque la implementación todavía
construye una lista. Python permite `len()` y el índice porque observa el objeto real;
mypy los rechaza porque solo puede garantizar lo que declara el contrato.

In [12]:
iterable_return_source = """from collections.abc import Iterable

def calculate_discounts(
    items: Iterable[float], discount: float
) -> list[float]:
    return [item * (1 - discount) for item in items]

discounted_prices = calculate_discounts([100.0, 200.0], 0.2)

count = len(discounted_prices)
first = discounted_prices[0]
"""
stdout, stderr, exit_status = mypy_api.run(
    ["--strict", "--no-incremental", "-c", iterable_return_source]
)
print(stdout)
assert exit_status == 0

runtime_namespace: dict[str, object] = {}
exec(iterable_return_source, runtime_namespace)
assert runtime_namespace["count"] == 2
assert runtime_namespace["first"] == 80.0

Success: no issues found in 1 source file



### Evitar `Any` como respuesta automática

`Any` permite saltarse muchas comprobaciones. Si un dato crudo puede ser cualquier
objeto, `object` obliga a comprobar su tipo antes de usar operaciones específicas.
Con `isinstance` podemos acotar el tipo. En los modelos, `Literal` limitará los
valores permitidos. Hoy no necesitamos genéricos avanzados ni protocolos.

In [13]:
def describe_input(value: object) -> str:
    if isinstance(value, str):
        return f"Text with {len(value)} characters"
    return "Not a text value"


print(describe_input("hello"))
print(describe_input(None))

Text with 5 characters
Not a text value


## 3. Un modelo de datos también es una clase

`BaseModel` es la clase base de Pydantic. Al heredar de ella y declarar campos,
obtenemos construcción, validación y serialización. Una instancia sigue teniendo
atributos: `prediction.score`, en lugar de `record["score"]`.

Comenzamos con un modelo deliberadamente pequeño para observar sus límites.

In [14]:
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator


class BasicPrediction(BaseModel):
    text: str
    score: float


basic_prediction = BasicPrediction.model_validate({"text": "Good service", "score": "0.9"})
print(basic_prediction)
print(type(basic_prediction.score))
assert basic_prediction.score == 0.9
# A float annotation alone does not constrain the numerical range.
print(BasicPrediction(text="Outside the range", score=1.5))

text='Good service' score=0.9
<class 'float'>
text='Outside the range' score=1.5


### Conversión y modo estricto

En modo habitual, Pydantic acepta algunas conversiones, como `"0.9"` a `0.9`.
**Convertir** no es lo mismo que **rechazar todo valor cuyo tipo original difiera**.
El modo estricto restringe conversiones; las restricciones de rango son otra regla.

Decidiremos el contrato según la fuente: un archivo puede contener números como
texto, mientras que una configuración interna puede exigir enteros reales.

In [15]:
try:
    BasicPrediction.model_validate({"text": "Good service", "score": "0.9"}, strict=True)
except ValidationError as error:
    print(error.errors()[0]["type"])
else:
    print("Strict validation should reject the string score")
    raise AssertionError("Strict validation should reject the string score")

float_type


# typing (módulo estándar)

## 4. Campos restringidos y errores útiles

`Literal` enumera etiquetas válidas; `Field` añade límites. `Annotated` combina
un tipo con metadatos: el alias `Score` se puede reutilizar sin repetir sus reglas.
`source: str | None = None` permite ausencia y también omitir el campo.

`extra="forbid"` rechaza claves desconocidas, como `scroe`; de otro modo un error
de escritura podría pasar inadvertido. Este primer modelo aún no normaliza texto.

In [16]:
from typing import Annotated, Literal

Label = Literal["positive", "neutral", "negative"]
Score = Annotated[float, Field(ge=0, le=1, allow_inf_nan=False)]


class ConstrainedPrediction(BaseModel):
    model_config = ConfigDict(extra="forbid")

    text: str = Field(min_length=1)
    label: Label
    score: Score
    source: str | None = None


constrained = ConstrainedPrediction(text="Good", label="positive", score=0.0)
assert constrained.source is None
print(constrained)

text='Good' label='positive' score=0.0 source=None


In [17]:
invalid_record = {"text": "", "label": "unknown", "score": 1.2, "scroe": 0.8}
try:
    ConstrainedPrediction.model_validate(invalid_record)
except ValidationError as error:
    for detail in error.errors():
        print(detail["loc"], detail["type"], detail["msg"])
else:
    raise AssertionError("The record should be rejected")

('text',) string_too_short String should have at least 1 character
('label',) literal_error Input should be 'positive', 'neutral' or 'negative'
('score',) less_than_equal Input should be less than or equal to 1
('scroe',) extra_forbidden Extra inputs are not permitted


Cada entrada de `errors()` describe un problema: `loc` indica el campo, `type`
su categoría y `msg` una explicación. Un registro puede producir varios errores.
La función que procesa un lote debe conservarlos, sin confundir número de errores
con número de registros rechazados.

- [x] terminado
### <span style="color: red;">Ejercicio 3 · Ausente, nulo e inválido</span>

- Prueba registros que omitan `source`,
- que usen `source=None`, que omitan `score`
- y que usen `score=None`.

- Predice cuáles se aceptan.
- Comprueba también límites 0 y 1.
Explica por qué `str | None` sin `= None` permitiría nulo pero seguiría siendo obligatorio.

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [18]:
# Build each in Omitir source: Se rechaza.
# Aunque acepte nulos (None), al no tener un valor por defecto (= None), el campo es obligatorio en el diccionario de entrada.

#Usar source=None: Se acepta. Al tiparse como str | None, el sistema permite explícitamente el valor nulo si la llave está presente.

#Omitir score: Se rechaza. Es un campo obligatorio y no se proporcionó.

#Usar score=None: Se rechaza (a menos que se tipara como float | None). Si es float, requiere un número.

#Límites 0 y 1 (score): Se aceptan los valores exactos 0.0 y 1.0 (asumiendo restricciones inclusivas ge=0, le=1). Valores como -0.01 o 1.01 son rechazados.put and catch only ValidationError for expected failures.
registros= [
    {"text": "Good", "label": "negative",  "score": 0.9},
    {"text": "Good", "label": "negative",  "score": None},
    {"text": "Bad", "label": "negative", "score":1.0},
    {"text": "Neutral","label": "neutral","score": None,"source":None},
    {"text": "Good", "label": "negative","score": 0.0, "source":None},
    {"text": "Good", "label": "negative","score": 1.0, "source":None}
]


# Omite source
try:
    for record in registros:
        ConstrainedPrediction.model_validate(record)
except ValidationError as e:
    for detail in e.errors():
        print(detail["loc"], detail["type"], detail["msg"])

('score',) float_type Input should be a valid number


## 5. Validadores: reglas que necesitan código

Queremos aceptar `" POSITIVE "` y rechazar textos de solo espacios. Antes de validar
los campos, quitaremos espacios externos y convertiremos las cadenas a minúsculas.
Es una decisión explícita de este taller, no una limpieza universal para todos los textos.

`@field_validator(..., mode="before")` registra una función que recibe la entrada
cruda. `@classmethod` indica que recibe la clase como `cls`, no una instancia como
`self`. Los decoradores conectan nuestra función con Pydantic; no necesitas crear
un decorador propio. Devuelve el valor transformado o lanza `ValueError`.

Los booleanos tienen un comportamiento numérico en Python; queremos rechazarlos
explícitamente como scores. La validación de rango rechazará también `NaN` e infinito.

In [19]:
class Prediction(BaseModel):
    """Normalize text fields and validate one incoming prediction."""

    model_config = ConfigDict(extra="forbid", validate_assignment=True)

    text: str = Field(min_length=1)
    label: Label
    score: Score
    source: str | None = None

    @field_validator("text", "label", "source", mode="before")
    @classmethod
    def normalize_strings(cls, value: object) -> object:
        if isinstance(value, str):
            return value.strip().lower()
        return value

    @field_validator("score", mode="before")
    @classmethod
    def reject_boolean_score(cls, value: object) -> object:
        if isinstance(value, bool):
            raise ValueError("A boolean is not a score")
        return value

In [20]:
normalized = Prediction.model_validate(
    {"text": "  GOOD SERVICE  ", "label": " POSITIVE ", "score": "0.9"}
)
assert normalized.text == "good service"
assert normalized.label == "positive"
print(normalized)

text='good service' label='positive' score=0.9 source=None


- [x] terminado
### <span style="color: red;"> Ejercicio 4 · Detectar los casos que faltaban</span>

- Prueba texto `"   "`,
- etiqueta `None`,
- score `True`,
- `"nan"`,
- `-0.1` y `1.1`.
Deben rechazarse.
- Comprueba también que `0.0` sea válido y que la entrada original
no se modifique al normalizarla.
- ¿Por qué `min_length=1` sin quitar espacios era insuficiente?

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [21]:
# Validate one changed field at a time and inspect the location of each error.
#{"texto":"caso de uso" ,"data": "payload"}

casos_invalidos = [
    {"texto": "Texto con solo espacios", "data": {"text": "   ", "label": "positive", "score": 0.5}},
    {"texto": "Etiqueta None", "data": {"text": "Good", "label": None, "score": 0.5}},
    {"texto": "Score booleano (True)", "data": {"text": "Good", "label": "positive", "score": True}},
    {"texto": "Score 'nan' (string)", "data": {"text": "Good", "label": "positive", "score": "nan"}},
    {"texto": "Score NaN (float)", "data": {"text": "Good", "label": "positive", "score": float("nan")}},
    {"texto": "Score menor a 0.0 (-0.1)", "data": {"text": "Good", "label": "positive", "score": -0.1}},
    {"texto": "Score mayor a 1.0 (1.1)", "data": {"text": "Good", "label": "positive", "score": 1.1}},
]

print("--- Comprobando rechazo de casos inválidos ---")
for caso in casos_invalidos:
  try:
    Prediction.model_validate(caso["data"])
  except ValidationError:
    print(f" Rechazado correctamente: {caso['texto']}")
  else:
    raise AssertionError(f" Error: El caso '{caso['texto']}' no fue rechazado.")


valido_cero = {"text": "Good", "label": "positive", "score": 0.0}
prediccion_cero = Prediction.model_validate(valido_cero)
assert prediccion_cero.score == 0.0, "El score 0.0 debe ser válido"
print(" Validado exitosamente: score 0.0")


entrada_original = {"text": " Servicio Excelente ", "label": "POSITIVE", "score": 0.95}
copia_entrada = entrada_original.copy()
objeto_creado = Prediction.model_validate(entrada_original)

assert entrada_original == copia_entrada, "El diccionario original no debe ser alterado"
assert entrada_original["text"] == " Servicio Excelente ", "El texto original conserva sus espacios"
assert objeto_creado.text == "servicio excelente", "El objeto de Pydantic sí fue normalizado"


--- Comprobando rechazo de casos inválidos ---
 Rechazado correctamente: Texto con solo espacios
 Rechazado correctamente: Etiqueta None
 Rechazado correctamente: Score booleano (True)
 Rechazado correctamente: Score 'nan' (string)
 Rechazado correctamente: Score NaN (float)
 Rechazado correctamente: Score menor a 0.0 (-0.1)
 Rechazado correctamente: Score mayor a 1.0 (1.1)
 Validado exitosamente: score 0.0


### Validar asignaciones tiene un alcance concreto

`validate_assignment=True` vuelve a validar al asignar atributos del modelo.
Sin esa opción, construir una instancia válida no garantiza que una asignación
posterior conserve el contrato. Esta opción no vigila cada modificación interna
de una lista: mantenemos las operaciones del lote en funciones conocidas.

In [22]:
try:
    normalized.score = 2.0
except ValidationError as error:
    print(error.errors()[0]["msg"])
assert normalized.score == 0.9

Input should be less than or equal to 1


## 6. Configuración explícita

La configuración también es dato: nombre del lote, umbral de selección y máximo
de registros. Aquí llega como diccionario local, sin variables de entorno ni secretos.

Distingue `model_config` (comportamiento de Pydantic) de `BatchConfig` (parámetros
de nuestra aplicación). El máximo será un entero estricto entre 1 y 1000: no
aceptaremos `"10"` ni `True`. El umbral comparte el contrato de los scores.

In [23]:
class BatchConfig(BaseModel):
    """Application settings supplied explicitly, without secrets or services."""

    model_config = ConfigDict(extra="forbid", validate_assignment=True)

    batch_name: str = Field(min_length=1)
    threshold: Score = 0.9
    max_records: int = Field(default=100, ge=1, le=1000, strict=True)

    @field_validator("batch_name")
    @classmethod
    def require_nonblank_name(cls, value: str) -> str:
        value = value.strip()
        if not value:
            raise ValueError("Batch name must not be blank")
        return value

    @field_validator("threshold", mode="before")
    @classmethod
    def reject_boolean_threshold(cls, value: object) -> object:
        if isinstance(value, bool):
            raise ValueError("A boolean is not a threshold")
        return value

In [24]:
config = BatchConfig.model_validate({"batch_name": " Demo batch ", "threshold": "0.9"})
assert config.batch_name == "Demo batch"
assert config.max_records == 100
print(config)

batch_name='Demo batch' threshold=0.9 max_records=100


- [x] terminado
### <span style="color: red;">Ejercicio 5 · Configuración antes de procesar</span>

Construye una configuración con nombre `"Practice"`, umbral 0 y máximo 3.
Después comprueba que máximo `"3"`, máximo 0 y nombre `"   "` produzcan errores.
Explica por qué un error de configuración debe detener el lote completo, mientras
que un registro inválido puede conservarse como rechazo.

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [25]:
# Create a valid configuration and try the invalid variants.
config = BatchConfig.model_validate({"batch_name": "Practice","threshold":0.0,"max_records":3})
assert config.batch_name == "Practice"
assert config.threshold == 0.0
assert config.max_records == 3
print(config)

try:
    BatchConfig.model_validate(
      {"batch_name": "Practice", "threshold": 0.0, "max_records": 0}
  )
except ValidationError:
  print("Rechazado correctamente: max_records=0 (viola ge=1)")
else:
  raise AssertionError("Debió fallar por ge=1 en max_records")

try:
  BatchConfig.model_validate(
      {"batch_name": "   ", "threshold": 0.0, "max_records": 3}
  )
except ValidationError:
  print("Rechazado correctamente: batch_name='   ' (nombre en blanco)")
else:
  raise AssertionError("Debió fallar por nombre en blanco")

batch_name='Practice' threshold=0.0 max_records=3
Rechazado correctamente: max_records=0 (viola ge=1)
Rechazado correctamente: batch_name='   ' (nombre en blanco)


### Registro de usuarios

El modelo usa:
 - `EmailStr` para validar el formato del correo
 - y `SecretStr` para ocultar la contraseña en su representación habitual.
 - `exclude=True` la excluye de la serialización.
 - `SecretStr` no cifra ni almacena de forma segura una contraseña.

El rol tiene un valor predeterminado válido. El ejemplo muestra la construcción
del modelo, sus errores de validación y su serialización.

In [26]:
from pydantic import EmailStr, SecretStr


class User(BaseModel):
    model_config = ConfigDict(extra="forbid")

    name: str = Field(min_length=2)
    email: EmailStr
    password: SecretStr = Field(exclude=True)
    role: Literal["author", "editor", "admin"] = "author"


user_data = {
    "name": "Alex",
    "email": "alex@example.com",
    "password": "ExamplePassword123",
}
user = User.model_validate(user_data)
print(user)
print(user.model_dump_json(indent=2))
assert user.role == "author"
assert "password" not in user.model_dump()

try:
    User.model_validate({**user_data, "email": "invalid-email"})
except ValidationError as error:
    print(error.errors()[0]["loc"], error.errors()[0]["msg"])
else:
    raise AssertionError("Invalid email was accepted")

name='Alex' email='alex@example.com' password=SecretStr('**********') role='author'
{
  "name": "Alex",
  "email": "alex@example.com",
  "role": "author"
}
('email',) value is not a valid email address: An email address must have an @-sign.


Un valor fuera del `Literal` y un campo ausente fallan por motivos distintos.
Comparamos `loc` y `type` para distinguir un valor inválido de un dato requerido.

In [27]:
try:
    User.model_validate({**user_data, "role": "developer"})
except ValidationError as error:
    invalid_role = error.errors()[0]
    print(invalid_role["loc"], invalid_role["type"], invalid_role["msg"])
else:
    raise AssertionError("Invalid role was accepted")

data_without_name = {key: value for key, value in user_data.items() if key != "name"}
try:
    User.model_validate(data_without_name)
except ValidationError as error:
    missing_name = error.errors()[0]
    print(missing_name["loc"], missing_name["type"], missing_name["msg"])
else:
    raise AssertionError("Missing name was accepted")

assert invalid_role["loc"] == ("role",)
assert invalid_role["type"] == "literal_error"
assert missing_name["loc"] == ("name",)
assert missing_name["type"] == "missing"

('role',) literal_error Input should be 'author', 'editor' or 'admin'
('name',) missing Field required


## 7. Modelos anidados y serialización

Un reporte contiene configuración, predicciones y errores: cada componente tiene
su propio modelo. `Field(default_factory=list)` crea una lista por instancia.
Una lista de modelos conserva estructura; no necesitamos diccionarios sin contrato
para pasar resultados entre funciones.

Serializar transforma la representación: `model_dump()` produce un diccionario y
`model_dump_json()` texto JSON. `model_validate_json()` reconstruye y valida desde JSON.
No equivale a escribir automáticamente un archivo ni a guardar en una base de datos.

In [28]:
class ValidationIssue(BaseModel):
    position: int
    field: str
    message: str
    error_type: str


class BatchReport(BaseModel):
    """ class report that inherits BaseModel """
    config: BatchConfig
    accepted: list[Prediction] = Field(default_factory=list)
    issues: list[ValidationIssue] = Field(default_factory=list)

In [29]:
first_empty_report = BatchReport(config=config)
second_empty_report = BatchReport(config=config)
first_empty_report.accepted.append(normalized)
assert second_empty_report.accepted == []

empty_report_06 = BatchReport(config=config, accepted=[normalized])
payload = empty_report_06.model_dump()
json_text = empty_report_06.model_dump_json(indent=2)
print(f"JSON: {json_text}")

assert isinstance(payload["accepted"][0], dict)
restored_report = BatchReport.model_validate_json(json_text)
assert restored_report == empty_report_06

JSON: {
  "config": {
    "batch_name": "Practice",
    "threshold": 0.0,
    "max_records": 3
  },
  "accepted": [
    {
      "text": "good service",
      "label": "positive",
      "score": 0.9,
      "source": null
    }
  ],
  "issues": []
}


- [x] terminado
### <span style="color: red;"> Ejercicio 6 · Ida y vuelta</span>

- Crea un reporte vacío,
- serialízalo a JSON
- reconstrúyelo.
- Comprueba igualdad y listas vacías.
- Después cambia el score a 5 en un diccionario
- obtenido con `example_report.model_dump()`:
- al validarlo debe aparecer la ubicación `("accepted", 0, "score")`.
- El modelo original debe conservar score 0.9.

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [30]:
# {"batch_name": "Practice", "threshold": 0.0, "max_records": 3}
config_new = BatchConfig.model_validate(
     {"batch_name": "name", "threshold": 0.0, "max_records": 1}
)

empty_report_06 = BatchReport(config=config_new, accepted=[normalized])

json_data = empty_report_06.model_dump_json(indent=2)
reconstructed_report = BatchReport.model_validate_json(json_data)

print(json_data)
print(reconstructed_report)
assert reconstructed_report == empty_report_06, (
    "El reporte reconstruido debe ser idéntico al original"
)

empty_report = BatchReport(config=config_new, accepted=[], issues=[])
assert empty_report.accepted == [], "La lista 'accepted' debe estar vacía"
assert empty_report.issues == [], "La lista 'issues' debe estar vacía"

data_dict = empty_report_06.model_dump()
data_dict["accepted"][0]["score"] = 5.0

try:
  BatchReport.model_validate(data_dict)
except ValidationError as error:
  first_error = error.errors()[0]
  error_loc = first_error["loc"]
  print(f"ValidationError capturado correctamente en la ubicación:"f" {error_loc}")

  assert error_loc == ("accepted", 0, "score"), (
      f"Ubicación esperada ('accepted', 0, 'score'), pero se obtuvo {error_loc}"
  )
else:
  raise AssertionError("Se esperaba un ValidationError al asignar score = 5.0")

assert empty_report_06.accepted[0].score == 0.9, (
    "El modelo original conserva el score 0.9"
)



{
  "config": {
    "batch_name": "name",
    "threshold": 0.0,
    "max_records": 1
  },
  "accepted": [
    {
      "text": "good service",
      "label": "positive",
      "score": 0.9,
      "source": null
    }
  ],
  "issues": []
}
config=BatchConfig(batch_name='name', threshold=0.0, max_records=1) accepted=[Prediction(text='good service', label='positive', score=0.9, source=None)] issues=[]
ValidationError capturado correctamente en la ubicación: ('accepted', 0, 'score')


## 8. De la notebook a un módulo reutilizable

La siguiente celda reúne las clases anteriores y
las funciones de procesamiento en `lesson_models.py`, el mismo archivo que acompaña
esta notebook en el repositorio. Así Colab funciona sin descargar archivos auxiliares.

**La celda escribe o reemplaza `lesson_models.py` en el directorio de trabajo.**
Si hiciste cambios propios en ese archivo, guárdalos con otro nombre antes de
volver a ejecutarla. `%%writefile` es una instrucción de Jupyter, no sintaxis Python.

Lee primero `process_records`: valida cada entrada, guarda cada problema con su
posición desde cero y no captura excepciones ajenas a validación. Si el lote supera
`max_records`, lanza `ValueError` antes de procesarlo. El umbral selecciona entre
registros válidos; no convierte en inválidas las predicciones con menor score.

Importamos el módulo como `models`: el prefijo distingue sus clases de las
versiones construidas durante la explicación y evita mezclar sus instancias.

In [31]:
%%writefile lesson_models.py
"""Typed contracts for synthetic prediction records; no model is trained here."""

from typing import Annotated, Literal

from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator

Label = Literal["positive", "neutral", "negative"]
Score = Annotated[float, Field(ge=0, le=1, allow_inf_nan=False)]


class Prediction(BaseModel):
    """Normalize text fields and validate one incoming prediction."""

    model_config = ConfigDict(extra="forbid", validate_assignment=True)

    text: str = Field(min_length=1)
    label: Label
    score: Score
    source: str | None = None

    @field_validator("text", "label", "source", mode="before")
    @classmethod
    def normalize_strings(cls, value: object) -> object:
        if isinstance(value, str):
            return value.strip().lower()
        return value

    @field_validator("score", mode="before")
    @classmethod
    def reject_boolean_score(cls, value: object) -> object:
        if isinstance(value, bool):
            raise ValueError("A boolean is not a score")
        return value


class BatchConfig(BaseModel):
    """Application settings supplied explicitly, without secrets or services."""

    model_config = ConfigDict(extra="forbid", validate_assignment=True)

    batch_name: str = Field(min_length=1)
    threshold: Score = 0.9
    max_records: int = Field(default=100, ge=1, le=1000, strict=True)

    @field_validator("batch_name")
    @classmethod
    def require_nonblank_name(cls, value: str) -> str:
        value = value.strip()
        if not value:
            raise ValueError("Batch name must not be blank")
        return value

    @field_validator("threshold", mode="before")
    @classmethod
    def reject_boolean_threshold(cls, value: object) -> object:
        if isinstance(value, bool):
            raise ValueError("A boolean is not a threshold")
        return value


class ValidationIssue(BaseModel):
    position: int
    field: str
    message: str
    error_type: str


class BatchReport(BaseModel):
    config: BatchConfig
    accepted: list[Prediction] = Field(default_factory=list)
    issues: list[ValidationIssue] = Field(default_factory=list)


def average_score(predictions: list[Prediction]) -> float | None:
    if not predictions:
        return None
    return sum(prediction.score for prediction in predictions) / len(predictions)


def process_records(records: list[object], config: BatchConfig) -> BatchReport:
    """Reject an oversized batch; preserve all validation issues for each row."""
    if len(records) > config.max_records:
        raise ValueError("Batch exceeds max_records")

    report = BatchReport(config=config)
    for position, record in enumerate(records):
        try:
            prediction = Prediction.model_validate(record)
        except ValidationError as error:
            for detail in error.errors():
                report.issues.append(
                    ValidationIssue(
                        position=position,
                        field=".".join(str(part) for part in detail["loc"]) or "<record>",
                        message=detail["msg"],
                        error_type=detail["type"],
                    )
                )
        else:
            report.accepted.append(prediction)
    return report


def select_predictions(report: BatchReport) -> list[Prediction]:
    """Select by confidence; this is not an accuracy measurement."""
    return [
        prediction
        for prediction in report.accepted
        if prediction.score >= report.config.threshold
    ]

Overwriting lesson_models.py


### Comprobar un archivo desde la terminal

Con el módulo ya guardado, el uso habitual es `mypy --strict lesson_models.py`.
El argumento final es el archivo que mypy revisará. No hace falta `-c` porque
el código ya está en un archivo, ni `--no-incremental` para el trabajo cotidiano.

La siguiente celda ejecuta el equivalente mediante el Python de esta notebook
y comprueba que termina sin errores.

In [32]:
import importlib
import subprocess
import lesson_models as models

# Reload after running the file-writing cell again.
models = importlib.reload(models)
from lesson_models import average_score, process_records, select_predictions

result = subprocess.run(
    [sys.executable, "-m", "mypy", "--strict", "lesson_models.py"],
    capture_output=True,
    text=True,
    check=False,
)
print(result.stdout or result.stderr)
assert result.returncode == 0, result.stderr or result.stdout

Success: no issues found in 1 source file



## 9. Práctica guiada: de entradas crudas a reporte

Estos seis registros fueron escritos para esta lección y se pueden reutilizar con
los materiales del curso. Incluyen dos válidos, un score fuera de rango, una
etiqueta nula, un score ausente y una entrada que no es un diccionario.

Antes de ejecutar: identifica posiciones válidas y rechazadas. Esperamos **2
aceptadas**, **4 posiciones rechazadas**, promedio **0.49** y **1 seleccionada** con
umbral 0.9. No calculamos exactitud: no existen etiquetas de referencia.

In [33]:
raw_records: list[object] = [
    {"text": " Excellent ", "label": " POSITIVE ", "score": "0.98"},
    {"text": "Zero is valid", "label": "neutral", "score": 0.0},
    {"text": "Out of range", "label": "negative", "score": 1.2},
    {"text": "Missing label", "label": None, "score": 0.8},
    {"text": "Missing score", "label": "neutral"},
    None,
]

batch_config = models.BatchConfig(batch_name="Class demo", threshold=0.9, max_records=10)
report = process_records(raw_records, batch_config)
rejected_positions = {issue.position for issue in report.issues}
print(f"Accepted: {len(report.accepted)}")
print(f"Rejected records: {len(rejected_positions)}")
print(f"Average score: {average_score(report.accepted)}")
print(f"Selected: {len(select_predictions(report))}")
for issue in report.issues:
    print(issue.position, issue.field, issue.error_type)

Accepted: 2
Rejected records: 4
Average score: 0.49
Selected: 1
2 score less_than_equal
3 label literal_error
4 score missing
5 <record> model_type


- [x] terminado
### <span style="color: red;">Ejercicio 7 · Comprobar antes de ampliar</span>

Verifica los resultados esperados del lote.
- Procesa una lista vacía
- y otra con solo `None`: sus promedios deben ser `None`.
- Prueba un lote mayor que el máximo: debe producir `ValueError`, sin truncarlo silenciosamente.
- Usa el módulo de apoyo.

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [49]:
config_test = models.BatchConfig(batch_name="Límites", threshold=0.5, max_records=5)

reporte_vacio = process_records([], config_test)
assert average_score(reporte_vacio.accepted) is None, "El promedio de una lista vacía debe ser None"
print(f"lista vacia: {average_score(reporte_vacio.accepted)}\n")
reporte_none = process_records([None], config_test)
assert average_score(reporte_none.accepted) is None, "El promedio de una lista con solo un elemento None debe ser None"
assert len(reporte_none.issues) == 1, "El elemento None debe generar un problema de validación"
print(f"resultado de promedio: {average_score(reporte_none.accepted)}\n")
print(f"problema de validacion: { reporte_none.issues}\n")


lote_excedido = [{"text": "Ok", "label": "neutral", "score": 0.5}]*3

# configuracion de limite
config_limite = models.BatchConfig(batch_name="Límite estricto", threshold=0.5, max_records=2)

try:
    process_records(lote_excedido, config_limite)
except ValueError as error:
    assert str(error) == "Batch exceeds max_records"
    print("[✓] Caso límite verificado: Se disparó ValueError correctamente al exceder max_records.")
else:
    raise AssertionError("No se bloqueó el lote que excedía max_records")




lista vacia: None

resultado de promedio: None

problema de validacion: [ValidationIssue(position=0, field='<record>', message='Input should be a valid dictionary or instance of Prediction', error_type='model_type')]

[✓] Caso límite verificado: Se disparó ValueError correctamente al exceder max_records.


- [x] terminado
### <span style="color: red;">Ejercicio 8 · Actividad independiente</span>

Escribe una función tipada `count_labels(predictions)`
- que devuelva un diccionario de conteos por etiqueta.
- Usa `list[models.Prediction]` como entrada
- y `dict[str, int]`como salida.
- Debe funcionar para lista vacía.

- Añádela a una copia de tu módulo, compruébala y ejecuta mypy sobre esa copia.
- Dentro de ese módulo usa `list[Prediction]`, porque la clase se define allí;
- el prefijo `models` se usa solo desde esta notebook.

- Como parte de la misma evidencia,
    -  crea un registro con **dos errores**,
    - conserva ambos y
    - explica por qué cuenta como **un registro rechazado**.
    - Serializa el reporte y comprueba que se pueda reconstruir.

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [53]:
# Implement count_labels and check it with validated predictions.
# Add the function to your own module copy after trying it here.
import lesson_models as models
from collections  import  Counter

def count_labels(predictions:list)->dict[str, int]:
  return dict(Counter(pred.label for pred in predictions))

config = models.BatchConfig(batch_name="Ejercicio 8", threshold=0.5, max_records=10)

assert count_labels([]) == {}, "Debe devolver un diccionario vacío si no hay predicciones"
print(f"Dict vacio: {count_labels([])}")

lote_multi_error = [
    {"text": "Excelente", "label": "positive", "score": 0.95}, # Posición 0: Válido
    {"text": "Error doble", "label": "inválido", "score": 1.5}  # Posición 1: 2 Errores
]

reporte = models.process_records(lote_multi_error, config)

posiciones_rechazadas = {issue.position for issue in reporte.issues}

assert len(reporte.issues) == 2, "Se deben registrar 2 fallas en report.issues"
assert len(posiciones_rechazadas) == 1, "Solo hay 1 registro rechazado (posición 1)"

json_str = reporte.model_dump_json(indent=2)
print(f"Serializado: {json_str}\n")
reporte_reconstruido = models.BatchReport.model_validate_json(json_str)
print(f"Deserializado : {reporte_reconstruido}\n")

assert reporte_reconstruido == reporte, "El reporte reconstruido debe ser idéntico al original"

print("[✓] count_labels funciona correctamente.")
print("[✓] Se capturaron 2 errores en 1 solo registro rechazado.")


Dict vacio: {}
Serializado: {
  "config": {
    "batch_name": "Ejercicio 8",
    "threshold": 0.5,
    "max_records": 10
  },
  "accepted": [
    {
      "text": "excelente",
      "label": "positive",
      "score": 0.95,
      "source": null
    }
  ],
  "issues": [
    {
      "position": 1,
      "field": "label",
      "message": "Input should be 'positive', 'neutral' or 'negative'",
      "error_type": "literal_error"
    },
    {
      "position": 1,
      "field": "score",
      "message": "Input should be less than or equal to 1",
      "error_type": "less_than_equal"
    }
  ]
}

Deserializado : config=BatchConfig(batch_name='Ejercicio 8', threshold=0.5, max_records=10) accepted=[Prediction(text='excelente', label='positive', score=0.95, source=None)] issues=[ValidationIssue(position=1, field='label', message="Input should be 'positive', 'neutral' or 'negative'", error_type='literal_error'), ValidationIssue(position=1, field='score', message='Input should be less than or equa

## 10. Una API pequeña con FastAPI y Pydantic

Una API puede recibir JSON cuyos valores no conocemos al escribir el programa.
FastAPI usa el modelo `QuoteRequest` para validar el cuerpo de la petición
antes de llamar a la función. Si los datos son válidos, la función recibe
una instancia con sus campos ya convertidos. `response_model` describe y
valida la respuesta.

`TestClient` enviará peticiones a la aplicación desde esta notebook, sin
iniciar un servidor. La organización de la aplicación como proyecto vendrá
en la sesión de uv.

In [36]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field


class QuoteRequest(BaseModel):
    item_name: str = Field(min_length=1)
    unit_price: float = Field(gt=0)
    quantity: int = Field(ge=1)


class QuoteResponse(BaseModel):
    item_name: str
    unit_price: float
    quantity: int
    total: float


app = FastAPI()


@app.post("/quotes", response_model=QuoteResponse)
def create_quote(quote: QuoteRequest) -> QuoteResponse:
    return QuoteResponse(
        item_name=quote.item_name,
        unit_price=quote.unit_price,
        quantity=quote.quantity,
        total=quote.unit_price * quote.quantity,
    )


client = TestClient(app)

### Una petición válida y otra que requiere conversión

En la segunda petición, `unit_price` llega como texto. Pydantic comprueba
que pueda convertirse a `float`; la función de la ruta recibe el número y
calcula el total. La respuesta JSON contiene un número, no el texto original.

In [37]:
valid_response = client.post(
    "/quotes",
    json={"item_name": "Notebook", "unit_price": 19.5, "quantity": 2},
)
converted_response = client.post(
    "/quotes",
    json={"item_name": "Notebook", "unit_price": "19.5", "quantity": 2},
)

assert valid_response.status_code == 200
assert converted_response.status_code == 200
assert valid_response.json() == converted_response.json()
assert converted_response.json()["total"] == 39.0
print(converted_response.json())

{'item_name': 'Notebook', 'unit_price': 19.5, 'quantity': 2, 'total': 39.0}


### Una petición rechazada

El texto `"nineteen"` no puede convertirse a `float` y una cantidad de
cero incumple la restricción `ge=1`. FastAPI responde con estado HTTP 422
y señala ambos campos. La función `create_quote` no se ejecuta para esta
petición.

In [38]:
invalid_response = client.post(
    "/quotes",
    json={"item_name": "Notebook", "unit_price": "nineteen", "quantity": 0},
)

assert invalid_response.status_code == 422
issues = invalid_response.json()["detail"]
assert {tuple(issue["loc"]) for issue in issues} == {
    ("body", "unit_price"),
    ("body", "quantity"),
}
for issue in issues:
    print(issue["loc"], issue["type"])

['body', 'unit_price'] float_parsing
['body', 'quantity'] greater_than_equal


## Evidencia y criterios

Conserva tu módulo con la función del ejercicio 8,
las comprobaciones y un reporte JSON. Puedes copiar la salida JSON o guardarla
localmente; todavía no se exige una base de datos ni un proyecto empaquetado.

- **Contratos claros:** parámetros y retornos anotados; ausencia de `Any` injustificado.
- **Validación:** restricciones y normalización justificadas; cero válido, faltantes y
  entradas inválidas distinguidos; configuración validada antes de procesar.
- **Errores útiles:** posición, campo y motivo conservados, incluidos varios errores por registro.
- **Serialización:** reconstrucción equivalente del reporte anidado.
- **Verificación:** mypy sin errores en el módulo final y comprobaciones de casos límite.

- [x] terminado
### <span style="color: red;">Profundización</span>

- Compara `model_dump(exclude_none=True)` con la salida habitual.

- ¿Cuál es la diferencia entre omitir `source` en la representación y cambiar el contrato del modelo?

## Cierre

Explica por qué una anotación no reemplaza la validación, qué información aporta
un error de Pydantic y qué decisiones de conversión tomaste. La próxima sesión
procesa datos de forma perezosa con iteradores, generadores y context managers.

## Referencias

- [Pydantic: modelos](https://docs.pydantic.dev/latest/concepts/models/).
- [Pydantic: validadores](https://docs.pydantic.dev/latest/concepts/validators/).
- [mypy: primeros pasos](https://mypy.readthedocs.io/en/stable/getting_started.html).
- [FastAPI: cuerpo de petición](https://fastapi.tiangolo.com/tutorial/body/).
- [FastAPI: TestClient](https://fastapi.tiangolo.com/tutorial/testing/).

- [PEP 1: propósito y proceso](https://peps.python.org/pep-0001/).
- [PEP 484: Type Hints](https://peps.python.org/pep-0484/).
